In [ ]:
# datasets.py

import os
import random
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


##############################################################
##################### Data and Masking #######################
##############################################################

# Equivalent to:
# setwd("~/fungva/working-dir/")
os.chdir(os.path.expanduser("~/fungva/working-dir/"))


def read_rds(path):
    """
    Placeholder for reading RDS files in Python.

    Recommended package:
        pip install pyreadr

    Then replace this function with:

        import pyreadr
        result = pyreadr.read_r(path)
        return next(iter(result.values()))

    The returned object should match the R object:
        fc_train.rds, fc_val.rds, fc_test.rds:
            list-like collection of 68 x 68 FC matrices

        covariates.rds:
            pandas DataFrame with RID and covariate columns
    """
    import pyreadr

    result = pyreadr.read_r(path)
    return next(iter(result.values()))


fc_train = read_rds("fc_train.rds")
fc_val = read_rds("fc_val.rds")
fc_test = read_rds("fc_test.rds")
cov_df = read_rds("covariates.rds")


def as_matrix_list(fc_obj):
    """
    Converts an RDS-loaded object into a list of numpy matrices.

    This handles common pyreadr return types:
    - list-like objects
    - dict-like objects
    - pandas DataFrame where each entry may be a matrix-like object

    You may need to adjust this depending on the exact RDS structure.
    """
    if isinstance(fc_obj, dict):
        return [np.asarray(v, dtype=np.float64) for v in fc_obj.values()]

    if isinstance(fc_obj, list):
        return [np.asarray(v, dtype=np.float64) for v in fc_obj]

    if isinstance(fc_obj, pd.DataFrame):
        # Case 1: each column is one subject/matrix object
        if fc_obj.shape[0] == 68 and fc_obj.shape[1] == 68:
            return [fc_obj.to_numpy(dtype=np.float64)]

        matrices = []
        for _, row in fc_obj.iterrows():
            arr = np.asarray(row, dtype=np.float64)
            if arr.size == 68 * 68:
                matrices.append(arr.reshape(68, 68))
        if len(matrices) > 0:
            return matrices

    raise TypeError(
        "Could not automatically convert fc object to a list of matrices. "
        "Inspect the object returned by pyreadr and adapt as_matrix_list()."
    )


fc_train = as_matrix_list(fc_train)
fc_val = as_matrix_list(fc_val)
fc_test = as_matrix_list(fc_test)

# R:
# fc_train_array = 1 - aperm(simplify2array(fc_train), c(3, 1, 2))
#
# In R, simplify2array(fc_train) gives shape:
#   68 x 68 x n_subjects
# aperm(..., c(3, 1, 2)) gives:
#   n_subjects x 68 x 68
#
# Python equivalent:
fc_train_array = 1.0 - np.stack(fc_train, axis=0)

# R:
# A_mat = apply(fc_train_array, c(2, 3), mean)
A_mat = np.mean(fc_train_array, axis=0)


def rank_first(row):
    """
    Equivalent to R:
        rank(row, ties.method = "first")

    Returns 1-based ranks.
    """
    order = np.argsort(row, kind="stable")
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(row) + 1)
    return ranks


# R:
# ranks = t(apply(A_mat, 1, function(row) rank(row, ties.method = "first")))
ranks = np.vstack([rank_first(row) for row in A_mat])

n_size = 2

masks = [
    (ranks < (n_size**1 + 1)).astype(np.float64),
    (ranks < (n_size**2 + 1)).astype(np.float64),
    (ranks < (n_size**3 + 1)).astype(np.float64),
    (ranks < (n_size**4 + 1)).astype(np.float64),
    (ranks < (n_size**5 + 1)).astype(np.float64),
    np.eye(2278, dtype=np.float64),
]

del fc_train_array, A_mat, ranks, n_size


def lt_rowwise(M):
    """
    Equivalent to the R function:

        lt_rowwise = function(M) {
          p = nrow(M)
          out = vector("numeric", (p * (p - 1)) / 2)
          k = 1L

          for (i in 2:p) {
            len = i - 1L
            out[k:(k + len - 1L)] = M[i, 1:(i - 1L)]
            k = k + len
          }

          out
        }

    This extracts the strict lower triangle row by row:
        M[1, 0],
        M[2, 0], M[2, 1],
        M[3, 0], M[3, 1], M[3, 2],
        ...
    """
    M = np.asarray(M, dtype=np.float64)
    p = M.shape[0]

    out = np.empty((p * (p - 1)) // 2, dtype=np.float64)
    k = 0

    for i in range(1, p):
        length = i
        out[k : k + length] = M[i, :i]
        k += length

    return out


# R:
# fc_train_matrix = do.call(rbind, lapply(fc_train, lt_rowwise))
fc_train_matrix = np.vstack([lt_rowwise(M) for M in fc_train])
fc_val_matrix = np.vstack([lt_rowwise(M) for M in fc_val])
fc_test_matrix = np.vstack([lt_rowwise(M) for M in fc_test])

eps = 1e-6

# R:
# atanh(pmin(pmax(fc_train_matrix, -1 + eps), 1 - eps))
fc_train_atanh = np.arctanh(np.clip(fc_train_matrix, -1 + eps, 1 - eps))
fc_val_atanh = np.arctanh(np.clip(fc_val_matrix, -1 + eps, 1 - eps))
fc_test_atanh = np.arctanh(np.clip(fc_test_matrix, -1 + eps, 1 - eps))


def get_subject_ids(fc_obj, fallback_prefix):
    """
    R uses:
        rownames(fc_train_atanh)

    In Python, row names are not automatically preserved after np.vstack.
    If your fc_train/fc_val/fc_test objects include subject IDs, extract them here.

    This fallback creates string IDs in the current order.
    You should replace this if your RDS files contain real RID names.
    """
    return np.array([str(i + 1) for i in range(len(fc_obj))])


train_ids = get_subject_ids(fc_train, "train")
val_ids = get_subject_ids(fc_val, "val")
test_ids = get_subject_ids(fc_test, "test")

num_vars = ["AGE", "EDUCATION", "pTau_ratio_log", "abeta_ratio_log"]
other_vars = ["SEX", "PARTNERED", "APOE4"]

# Ensure RID is string-like for matching, since rownames from R are usually strings.
cov_df = cov_df.copy()
cov_df["RID"] = cov_df["RID"].astype(str)


def align_covariates(cov_df, ids):
    """
    Equivalent to:

        cov_df %>%
          filter(RID %in% ids) %>%
          slice(match(ids, RID))

    This preserves the exact order of ids.
    """
    ids = pd.Index(ids.astype(str), name="RID")

    aligned = (
        cov_df[cov_df["RID"].isin(ids)]
        .set_index("RID")
        .loc[ids]
        .reset_index()
    )

    return aligned


cov_train = align_covariates(cov_df, train_ids)
cov_val = align_covariates(cov_df, val_ids)
cov_test = align_covariates(cov_df, test_ids)

# R:
# train_means = sapply(cov_train[num_vars], mean, na.rm = TRUE)
# train_sds   = sapply(cov_train[num_vars], sd, na.rm = TRUE)
#
# Important:
# R's sd() uses sample standard deviation, ddof = 1.
train_means = cov_train[num_vars].mean(skipna=True)
train_sds = cov_train[num_vars].std(skipna=True, ddof=1)


def scale_with_train(df, vars, means, sds):
    """
    Equivalent to:

        x = as.matrix(df[, vars, drop = FALSE])
        x = sweep(x, 2, means, "-")
        x = sweep(x, 2, sds, "/")
    """
    x = df[vars].to_numpy(dtype=np.float64)
    x = x - means.to_numpy(dtype=np.float64)
    x = x / sds.to_numpy(dtype=np.float64)
    return x


cov_num_train = scale_with_train(cov_train, num_vars, train_means, train_sds)
cov_num_val = scale_with_train(cov_val, num_vars, train_means, train_sds)
cov_num_test = scale_with_train(cov_test, num_vars, train_means, train_sds)

cov_other_train = cov_train[other_vars].to_numpy(dtype=np.float64)
cov_other_val = cov_val[other_vars].to_numpy(dtype=np.float64)
cov_other_test = cov_test[other_vars].to_numpy(dtype=np.float64)

cov_train_mat = np.column_stack([cov_num_train, cov_other_train]).astype(np.float64)
cov_val_mat = np.column_stack([cov_num_val, cov_other_val]).astype(np.float64)
cov_test_mat = np.column_stack([cov_num_test, cov_other_test]).astype(np.float64)

cov_dim = cov_train_mat.shape[1]

fc_train_atanh = fc_train_atanh.astype(np.float64)
fc_val_atanh = fc_val_atanh.astype(np.float64)
fc_test_atanh = fc_test_atanh.astype(np.float64)
cov_train_mat = cov_train_mat.astype(np.float64)
cov_val_mat = cov_val_mat.astype(np.float64)
cov_test_mat = cov_test_mat.astype(np.float64)


##############################################################
######################## Dataset #############################
##############################################################

class FungvaDataset(Dataset):
    """
    Equivalent to fungva_dataset(), fungva_getitem(), and fungva_length().
    """

    def __init__(self, fc_mat, cov_mat):
        if fc_mat.shape[0] != cov_mat.shape[0]:
            raise ValueError("fc_mat and cov_mat must have the same number of rows.")

        self.fc_mat = torch.tensor(fc_mat, dtype=torch.float64)
        self.cov_mat = torch.tensor(cov_mat, dtype=torch.float64)
        self.n = fc_mat.shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        fc = self.fc_mat[i]
        cov = self.cov_mat[i]

        return {
            "x": {
                "fc": fc,
                "cov": cov,
            },
            "y": fc,
            "indices": i,
        }


def collate_fungva_batch(batch):
    """
    Makes PyTorch batches match your R dataloader structure:

        list(
          x = list(
            fc = batch_fc,
            cov = batch_cov
          ),
          y = batch_fc,
          indices = batch_idx
        )
    """
    fc = torch.stack([item["x"]["fc"] for item in batch], dim=0)
    cov = torch.stack([item["x"]["cov"] for item in batch], dim=0)
    y = torch.stack([item["y"] for item in batch], dim=0)
    indices = torch.tensor([item["indices"] for item in batch], dtype=torch.long)

    return {
        "x": {
            "fc": fc,
            "cov": cov,
        },
        "y": y,
        "indices": indices,
    }


train_ds = FungvaDataset(fc_train_atanh, cov_train_mat)
val_ds = FungvaDataset(fc_val_atanh, cov_val_mat)
test_ds = FungvaDataset(fc_test_atanh, cov_test_mat)

# R:
# set.seed(1)
random.seed(1)
np.random.seed(1)
torch.manual_seed(1)

train_dl = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fungva_batch,
)

val_dl = DataLoader(
    val_ds,
    batch_size=15,
    shuffle=False,
    collate_fn=collate_fungva_batch,
)

test_dl = DataLoader(
    test_ds,
    batch_size=15,
    shuffle=False,
    collate_fn=collate_fungva_batch,
)